# NpuKit — MCU-class DS-CNN MNIST peer

**Role:** TinyML depthwise-separable CNN shaped like what you’d run on a Cortex-M / TFLite Micro–class MCU.

- Host-only int8 — **not** on the FPGA, **not** a ViT stem
- Compare to tiny-ViT as **MCU/MPU + NpuKit accelerator** peer
- Headline: accuracy + weight KiB + where compute runs (not equal params/layers)

```bash
python3 host/train_dscnn_mnist.py
python3 host/dscnn_mnist.py --rebuild-int8
```


In [1]:
import importlib
import sys
from pathlib import Path

HOST = Path(".").resolve()
if not (HOST / "dscnn_mnist.py").exists():
    HOST = Path("/home/user/fpga/npukit/host")
sys.path.insert(0, str(HOST))

import dscnn_mnist as dscnn

importlib.reload(dscnn)
print("weights", dscnn.WEIGHTS_PATH, "exists", dscnn.WEIGHTS_PATH.exists())
print("int8   ", dscnn.INT8_PATH, "exists", dscnn.INT8_PATH.exists())
print("metrics", dscnn.METRICS_PATH, "exists", dscnn.METRICS_PATH.exists())

weights /home/user/fpga/npukit/host/dscnn_mnist_weights.pt exists True
int8    /home/user/fpga/npukit/host/dscnn_mnist_int8.npz exists True
metrics /home/user/fpga/npukit/host/dscnn_mnist_metrics.json exists True


## Eval full MNIST test (float + int8) vs ViT

In [2]:
assert dscnn.WEIGHTS_PATH.exists(), "run: python3 host/train_dscnn_mnist.py"
m = dscnn.run_eval(batch=256)
print()
print("=== summary ===")
print(f"DS-CNN float test: {100 * m.test_acc_float:.2f}%")
print(f"DS-CNN int8  test: {100 * m.test_acc_int8:.2f}%  (fair peer)")
print(f"params={m.n_params}  n_test={m.n_test}  qat_epochs={m.qat_epochs}")
print(dscnn.compare_vit_line())
print("PASS: DS-CNN host float+int8 reference metrics recorded")

=== DS-CNN MNIST host reference ===
device=cpu  params=9034
test accuracy (float, full 10000): 98.00%
test accuracy (int8,  full 10000): 98.39%
ViT (deploy-quantized numpy, full 10k): ~94.28%  [weights: vit_mnist_weights.npz]
NOTE: DS-CNN is a separate benchmark model (not a ViT stem, not on FPGA).
Fair compare: DS-CNN int8 vs ViT deploy-quant (~94.28%).
wrote /home/user/fpga/npukit/host/dscnn_mnist_metrics.json

=== summary ===
DS-CNN float test: 98.00%
DS-CNN int8  test: 98.39%  (fair peer)
params=9034  n_test=10000  qat_epochs=3
ViT (deploy-quantized numpy, full 10k): ~94.28%  [weights: vit_mnist_weights.npz]
PASS: DS-CNN host float+int8 reference metrics recorded
